# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv("data/AviationData.csv", encoding="latin1")
df.head()

C:\Users\mogou\AppData\Local\Temp\ipykernel_13204\4282342611.py:1: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/AviationData.csv", encoding="latin1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
df.shape

(88889, 31)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [5]:
df.isna().sum().sort_values(ascending=False).head(30)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
dtype: int64

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

df = df[df["Event.Date"].dt.year >= 1983].copy()

df.shape

(85289, 31)

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [7]:
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

df[injury_cols] = df[injury_cols].fillna(0)

df["Total.Passengers"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"] +
    df["Total.Minor.Injuries"] +
    df["Total.Uninjured"]
)

df["Fatal.Serious.Injuries"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"]
)

df = df[df["Total.Passengers"] > 0].copy()

df["Fatal.Serious.Injury.Rate"] = (
    df["Fatal.Serious.Injuries"] / df["Total.Passengers"]
)

df[[
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Total.Passengers",
    "Fatal.Serious.Injury.Rate"
]].head()

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.Passengers,Fatal.Serious.Injury.Rate
3600,0.0,1.0,0.0,1.0,2.0,0.5
3601,0.0,0.0,1.0,3.0,4.0,0.0
3602,0.0,0.0,0.0,2.0,2.0,0.0
3603,0.0,0.0,0.0,1.0,1.0,0.0
3604,0.0,0.0,2.0,0.0,2.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [8]:
df["Aircraft.damage"].value_counts(dropna=False)

Aircraft.damage
Substantial    61506
Destroyed      17479
NaN             2534
Minor           2369
Unknown           92
Name: count, dtype: int64

In [9]:
df["Aircraft.damage"] = df["Aircraft.damage"].str.strip().str.title()

df["Aircraft.Destroyed"] = np.where(
    df["Aircraft.damage"] == "Destroyed",
    1,
    0
)

df[["Aircraft.damage", "Aircraft.Destroyed"]].head()

,Aircraft.damage,Aircraft.Destroyed
3600,NaN,0
3601,Substantial,0
3602,Substantial,0
3603,Substantial,0
3604,Substantial,0


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [10]:
df["Make"].isna().sum()

np.int64(34)

In [11]:
df = df.dropna(subset=["Make"]).copy()

df["Make"] = df["Make"].str.upper().str.strip()

df["Make"].value_counts().head(20)

Make
CESSNA               25712
PIPER                14101
BEECH                 5082
BELL                  2574
BOEING                2097
MOONEY                1275
ROBINSON              1185
GRUMMAN               1068
BELLANCA               982
HUGHES                 878
SCHWEIZER              753
AIR TRACTOR            668
AERONCA                606
MAULE                  571
MCDONNELL DOUGLAS      560
CHAMPION               507
STINSON                421
AERO COMMANDER         411
DE HAVILLAND           400
LUSCOMBE               391
Name: count, dtype: int64

In [12]:
make_counts = df["Make"].value_counts()

valid_makes = make_counts[make_counts >= 50].index

df = df[df["Make"].isin(valid_makes)].copy()

df["Make"].value_counts().head(20)

Make
CESSNA               25712
PIPER                14101
BEECH                 5082
BELL                  2574
BOEING                2097
MOONEY                1275
ROBINSON              1185
GRUMMAN               1068
BELLANCA               982
HUGHES                 878
SCHWEIZER              753
AIR TRACTOR            668
AERONCA                606
MAULE                  571
MCDONNELL DOUGLAS      560
CHAMPION               507
STINSON                421
AERO COMMANDER         411
DE HAVILLAND           400
LUSCOMBE               391
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [13]:
df["Model"].isna().sum()

np.int64(16)

In [14]:
df = df.dropna(subset=["Model"]).copy()

df["Model"] = df["Model"].astype(str).str.upper().str.strip()

df["Model"].value_counts().head(20)

Model
152          2226
172          1641
172N         1093
PA-28-140     868
172M          759
150           748
172P          663
182           615
180           596
150M          553
PA-18-150     550
PA-28-180     548
PA-18         548
PA-28-161     526
PA-28-181     508
206B          486
150L          436
A36           428
PA-38-112     425
G-164A        423
Name: count, dtype: int64

In [15]:
df["Make.Model"] = df["Make"] + " " + df["Model"]

df["Make.Model"].value_counts().head(20)

Make.Model
CESSNA 152         2226
CESSNA 172         1641
CESSNA 172N        1093
PIPER PA-28-140     868
CESSNA 172M         759
CESSNA 150          748
CESSNA 172P         663
CESSNA 182          615
CESSNA 180          595
CESSNA 150M         553
PIPER PA-18-150     550
PIPER PA-28-180     548
PIPER PA-18         548
PIPER PA-28-161     526
PIPER PA-28-181     508
BELL 206B           485
CESSNA 150L         436
PIPER PA-38-112     425
BEECH A36           408
CESSNA 140          384
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [16]:
cols_to_clean = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

for col in cols_to_clean:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace("Nan", np.nan)

df[cols_to_clean].head()

,Engine.Type,Weather.Condition,Purpose.of.flight,Broad.phase.of.flight
3601,Reciprocating,Vmc,Personal,Approach
3602,Reciprocating,Vmc,Personal,Landing
3603,Reciprocating,Vmc,Personal,Takeoff
3604,Reciprocating,Vmc,Personal,Approach
3605,Reciprocating,Vmc,Instructional,Landing


In [17]:
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")

df["Number.of.Engines"].value_counts(dropna=False).sort_index()

Number.of.Engines
0.0      722
1.0    55027
2.0     9443
3.0      427
4.0      379
8.0        1
NaN     4163
Name: count, dtype: int64

In [18]:
df["Aircraft.Size"] = np.where(
    df["Total.Passengers"] <= 10,
    "Small Aircraft",
    "Large Aircraft"
)

df["Aircraft.Size"].value_counts()

Aircraft.Size
Small Aircraft    67148
Large Aircraft     3014
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [19]:
missing_percent = df.isna().mean().sort_values(ascending=False)

missing_percent.head(30)

Schedule                  0.852413
Air.carrier               0.816753
FAR.Description           0.702389
Aircraft.Category         0.700607
Longitude                 0.627947
Latitude                  0.627875
Airport.Code              0.430133
Airport.Name              0.403652
Broad.phase.of.flight     0.280893
Publication.Date          0.170990
Engine.Type               0.067230
Purpose.of.flight         0.063239
Report.Status             0.061828
Number.of.Engines         0.059334
Weather.Condition         0.040578
Aircraft.damage           0.030871
Registration.Number       0.014352
Country                   0.002737
Amateur.Built             0.000770
Location                  0.000570
Event.Id                  0.000000
Event.Date                0.000000
Accident.Number           0.000000
Investigation.Type        0.000000
Injury.Severity           0.000000
Make                      0.000000
Model                     0.000000
Total.Serious.Injuries    0.000000
Total.Fatal.Injuries

In [20]:
cols_to_drop = missing_percent[missing_percent > 0.80].index

df_clean = df.drop(columns=cols_to_drop).copy()

df_clean.shape

(70162, 35)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [21]:
df_clean.to_csv("data/aviation_cleaned.csv", index=False)

print("Cleaned data saved successfully.")
print(df_clean.shape)

Cleaned data saved successfully.
(70162, 35)
